# 第 14 天：因子中性化

> 来自《30 天因子研究计划》第 14 天  
> 主题：因子中性化  
> 必做：行业中性化  
> 选做：市值中性化  
> 目标产出：中性化模块

---

## 0. 今天你要真正学会什么？

因子中性化是在问：


这个因子真的在表达自己的信息，还是只是行业或市值暴露？


今天掌握：

1. 为什么需要行业中性化。
2. 为什么需要市值中性化。
3. 如何用回归残差做中性化。
4. 如何封装一个中性化模块。

---

## 1. 中性化直觉

假设一个价值因子高分股票大多是银行。  
如果高分组跑赢，可能是价值因子有效，也可能只是银行行业上涨。

中性化就是把行业、市值等已知暴露剥离掉，让因子更接近自身信息。

---

## 2. 回归残差法

用回归表达：


raw_factor = industry_effect + size_effect + residual


中性化后的因子就是残差：


neutral_factor = residual


---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260706)


---

## 4. 构造模拟截面数据


In [ ]:
n = 500
industries = ["金融", "消费", "科技", "医药", "周期"]

data = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "industry": rng.choice(industries, size=n),
    "log_market_cap": rng.normal(10, 1.1, size=n),
})

industry_effect_map = {"金融": 0.8, "消费": 0.3, "科技": -0.2, "医药": 0.1, "周期": -0.4}
data["raw_factor"] = (
    data["industry"].map(industry_effect_map)
    - 0.35 * data["log_market_cap"]
    + rng.normal(0, 0.7, size=n)
)

data.head()


---

## 5. 看中性化前的暴露


In [ ]:
industry_mean_before = data.groupby("industry")["raw_factor"].mean()
size_corr_before = data["raw_factor"].corr(data["log_market_cap"])

industry_mean_before, size_corr_before


如果行业均值差异很大，说明因子有明显行业暴露。  
如果和市值相关性很高，说明因子带有市值暴露。

---

## 6. 写中性化函数


In [ ]:
def neutralize_factor(
    df: pd.DataFrame,
    factor_col: str,
    industry_col: str | None = None,
    size_col: str | None = None,
) -> pd.Series:
    y = df[factor_col].astype(float)
    x_parts = [pd.Series(1.0, index=df.index, name="intercept")]

    if industry_col is not None:
        dummies = pd.get_dummies(df[industry_col], prefix="ind", drop_first=True).astype(float)
        x_parts.append(dummies)

    if size_col is not None:
        x_parts.append(df[[size_col]].astype(float))

    X = pd.concat(x_parts, axis=1)
    valid = pd.concat([y, X], axis=1).dropna()

    coef = np.linalg.lstsq(valid[X.columns].values, valid[factor_col].values, rcond=None)[0]
    fitted = pd.Series(valid[X.columns].values @ coef, index=valid.index)
    residual = valid[factor_col] - fitted

    out = pd.Series(np.nan, index=df.index, name=f"{factor_col}_neutral")
    out.loc[valid.index] = residual
    return out


data["factor_ind_neutral"] = neutralize_factor(data, "raw_factor", industry_col="industry")
data["factor_ind_size_neutral"] = neutralize_factor(data, "raw_factor", industry_col="industry", size_col="log_market_cap")
data.head()


---

## 7. 检查中性化效果


In [ ]:
check = pd.DataFrame({
    "before": data.groupby("industry")["raw_factor"].mean(),
    "industry_neutral": data.groupby("industry")["factor_ind_neutral"].mean(),
    "industry_size_neutral": data.groupby("industry")["factor_ind_size_neutral"].mean(),
})

size_corr_after = {
    "before": data["raw_factor"].corr(data["log_market_cap"]),
    "industry_neutral": data["factor_ind_neutral"].corr(data["log_market_cap"]),
    "industry_size_neutral": data["factor_ind_size_neutral"].corr(data["log_market_cap"]),
}

check, size_corr_after


中性化后，行业均值应更接近 0。  
市值中性化后，因子和市值相关性也应更接近 0。

---

## 8. 多日期中性化

真实因子表通常是长表：`date, ticker, factor, industry, log_market_cap`。  
中性化必须每天单独做。


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=20)
panel = []
for date in dates:
    temp = data.copy()
    temp["date"] = date
    temp["raw_factor"] = temp["raw_factor"] + rng.normal(0, 0.2, size=len(temp))
    panel.append(temp)
panel = pd.concat(panel, ignore_index=True)

panel["factor_neutral"] = panel.groupby("date", group_keys=False).apply(
    lambda g: neutralize_factor(g, "raw_factor", industry_col="industry", size_col="log_market_cap"),
    include_groups=False
)

panel.head()


---

## 9. 目标产出：中性化模块


In [ ]:
def neutralize_panel(
    df: pd.DataFrame,
    factor_col: str,
    date_col: str = "date",
    industry_col: str | None = "industry",
    size_col: str | None = "log_market_cap",
    output_col: str = "factor_neutral",
) -> pd.DataFrame:
    out = df.copy()
    out[output_col] = out.groupby(date_col, group_keys=False).apply(
        lambda g: neutralize_factor(g, factor_col, industry_col=industry_col, size_col=size_col),
        include_groups=False
    )
    return out


neutralized_panel = neutralize_panel(panel, "raw_factor")
neutralized_panel[["date", "ticker", "raw_factor", "factor_neutral"]].head()


---

## 10. 知识图谱


In [ ]:
mindmap
  root((因子中性化))
    行业中性化
      行业哑变量
      剥离行业均值
    市值中性化
      log市值
      剥离规模暴露
    方法
      回归
      残差
      每日截面
    检查
      行业均值
      市值相关
      IC变化


---

## 11. 作业

1. 只做行业中性化，观察市值相关性是否仍存在。
2. 只做市值中性化，观察行业均值是否仍不同。
3. 把中性化前后因子都计算一次 IC。
4. 思考：中性化会不会把有效信息也剥离掉？

---

## 12. 自测题

1. 中性化的核心思想是什么？  
   答案：剥离行业、市值等已知暴露，保留残差信息。

2. 为什么要每天单独中性化？  
   答案：每天的股票池、行业分布和市值结构都不同。

3. 中性化后一定更好吗？  
   答案：不一定，可能剥离掉一部分有效信息，需要检验。

---

## 13. 明天预告

明天学习去极值和标准化，搭建因子预处理流水线。

---

## 14. 仅供学习的提醒

本文使用模拟数据解释因子中性化方法，不构成任何投资建议。

---

# 统一高质量增强模块

> 本增强模块用于把第 14 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：因子中性化
- 必做：行业中性化
- 选做：市值中性化
- 目标产出：中性化模块

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

高价值组全是银行，低价值组全是科技，回测赚钱到底是价值有效还是行业轮动？中性化就是剥离这些已知暴露。

这个例子背后的关键直觉是：

> 中性化不是装饰，而是确认因子到底在表达什么。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


因子中性化
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 中性化模块


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(114)
n = 300
df = pd.DataFrame({
    "industry": rng.choice(["金融", "消费", "科技", "周期"], n),
    "log_mcap": rng.normal(10, 1, n),
})
effect = {"金融": .8, "消费": .2, "科技": -.3, "周期": -.5}
df["factor"] = df["industry"].map(effect) - .4 * df["log_mcap"] + rng.normal(0, .8, n)
X = pd.concat([pd.Series(1, index=df.index, name="intercept"), pd.get_dummies(df["industry"], drop_first=True).astype(float), df[["log_mcap"]]], axis=1)
coef = np.linalg.lstsq(X.values, df["factor"].values, rcond=None)[0]
df["factor_neutral"] = df["factor"] - X.values @ coef
print("size_corr_before:", round(df["factor"].corr(df["log_mcap"]), 4))
print("size_corr_after:", round(df["factor_neutral"].corr(df["log_mcap"]), 4))
print(df.groupby("industry")[["factor", "factor_neutral"]].mean().round(4))


## E. 产出验收标准

完成今天课程后，你的 `中性化模块` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `因子中性化` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `中性化模块` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `中性化模块`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 14 天复盘：因子中性化

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
---

# 第 10-15 天深度加厚模块


    ## L. 为什么还要加厚这一课？

    这一课属于第 10-15 天的“工程化因子”部分：它不像 Alpha、Beta 那样只靠概念就能建立直觉，也不像 PE、ROE 那样有明确财务含义。它更依赖窗口、参数、预处理顺序和检验口径。

    所以学习 `因子中性化` 时，不能只停留在“知道公式”。你至少要完成三层理解：


    第一层：公式能写对
    第二层：参数变化后结果还能解释
    第三层：能放进统一因子流水线


    如果只学第一层，代码很快能写出来，但研究时很容易陷入“参数换一下结果就变”的困境。

    ## M. 更贴近真实研究的场景

    价值因子可能天然偏金融，质量因子可能天然偏消费，动量因子可能阶段性偏小盘。不中性化，你可能以为自己买的是因子，其实买的是行业或市值。

    这类问题在真实研究中很常见：一个因子看似简单，但只要换股票池、换窗口、换持有期、换市场阶段，结果就会明显变化。成熟的研究方式不是逃避这种变化，而是把变化记录下来、解释出来。

    ## N. 参数敏感性实验

    下面这段代码是专门为本课补充的参数实验。它不追求复杂，而是训练一个习惯：

    > 不要只交一个因子结果，至少比较几组合理参数。


In [ ]:
    import numpy as np
import pandas as pd

rng = np.random.default_rng(214)
n = 500
df = pd.DataFrame({
    "industry": rng.choice(["金融","消费","科技","周期","医药"], n),
    "log_mcap": rng.normal(10, 1.0, n),
})
ind_effect = {"金融": .8, "消费": .3, "科技": -.2, "周期": -.5, "医药": .1}
df["factor"] = df["industry"].map(ind_effect) - .35*df["log_mcap"] + rng.normal(0,.8,n)
df["future_ret"] = .01*df["factor"] + rng.normal(0,.05,n)
X = pd.concat([pd.Series(1,index=df.index,name="intercept"), pd.get_dummies(df["industry"], drop_first=True).astype(float), df[["log_mcap"]]], axis=1)
beta = np.linalg.lstsq(X.values, df["factor"].values, rcond=None)[0]
df["factor_neutral"] = df["factor"] - X.values @ beta
print("raw_size_corr", round(df["factor"].corr(df["log_mcap"]),4))
print("neutral_size_corr", round(df["factor_neutral"].corr(df["log_mcap"]),4))
print("raw_ic", round(df["factor"].corr(df["future_ret"], method="spearman"),4))
print("neutral_ic", round(df["factor_neutral"].corr(df["future_ret"], method="spearman"),4))


    ## O. 结果该怎么写进研究笔记？

    建议你用下面这个格式记录：


    因子名称：因子中性化

    1. 使用的数据：
       - 股票池：
       - 时间区间：
       - 价格 / 财务口径：

    2. 核心参数：
       - 主参数：
       - 对照参数：

    3. 因子方向：
       - 因子值越大代表：
       - 是否需要取负号：

    4. 检验结果：
       - Rank IC：
       - ICIR：
       - 分组收益：
       - 多空表现：

    5. 稳定性：
       - 参数变化后是否稳定：
       - 分阶段是否稳定：
       - 极端行情是否失效：

    6. 结论：
       - 是否进入因子库：
       - 还需要什么后续验证：


    ## P. 额外验收清单

    `中性化模块` 如果要达到可复用标准，额外检查：

    1. 中性化前后都要计算 IC。
2. 检查行业均值是否接近 0。
3. 检查与 log 市值相关是否下降。
4. 说明是否可能剥离有效信息。

    ## Q. 更深入的常见误区

    ### 误区 1：参数越多越高级

    参数多不代表研究深，很多时候只是过拟合空间更大。真正高级的是解释参数为什么合理，并证明它在相邻参数下仍然不崩。

    ### 误区 2：只看全样本平均

    全样本平均可能掩盖阶段失效。至少要分年度、分市场状态、分股票池看一次。

    ### 误区 3：把预处理当成机械步骤

    去极值、标准化、中性化会改变因子含义。每加一步，都要知道自己剥离了什么，也可能损失了什么。

    ### 误区 4：忽略交易可行性

    技术、波动率、流动性类因子往往换手更高，交易成本可能非常关键。纸面有效不等于可交易。

    ## R. 加厚作业

    1. 把本课主参数上下各调整一次，记录结果变化。
    2. 把未来收益标签从 20 日改成 5 日和 60 日，观察结论是否变。
    3. 随机删除 10% 股票样本，检查结果是否稳定。
    4. 把因子取反，确认分组结果是否镜像变化。
    5. 写一段 200 字研究结论，必须同时包含“支持证据”和“风险提示”。

    ## S. 一句话升级结论

    `因子中性化` 的高质量学习标准不是“会算”，而是：

    > 会定义、会检验、会解释参数变化，也知道它在真实交易里可能被什么击穿。
